# CNN Training for Self-Driving RC Car Steering Control

This notebook trains a Convolutional Neural Network (CNN) to predict the steering angle of a small RC car based on camera images.
The trained model outputs a normalized steering value in the range [-1, 1], which is later used for real-time inference and closed-loop PID steering control on an Arduino.


# (1) Setup & Imports

In [ ]:
# If this runs, your notebook is working
print("Notebook running ✅")

import os
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional: if using OpenCV for image loading
import cv2

# PyTorch (for CNN)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# (2) Check GPU

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))

# (3) Define dataset paths

In [ ]:
from pathlib import Path

# Put your dataset folder beside the notebook, like:
# CNN-Self-Driving-Car/data/driving_log.csv
# CNN-Self-Driving-Car/data/IMG/

DATA_DIR = Path("data").resolve()
CSV_PATH = DATA_DIR / "driving_log.csv"
IMG_DIR  = DATA_DIR / "IMG"

print("DATA_DIR:", DATA_DIR)
print("CSV exists:", CSV_PATH.exists())
print("IMG exists:", IMG_DIR.exists())

# (4) Load CSV + Fix column

In [ ]:
import pandas as pd
import os

# 1) Load CSV
df_raw = pd.read_csv(CSV_PATH)
print("CSV columns:", df_raw.columns.tolist())
print("Rows:", len(df_raw))

# 2) Pick image column (common: 'center' or 'image_path')
possible_img_cols = ["image_path", "center", "img", "image", "filename"]
img_col = next((c for c in possible_img_cols if c in df_raw.columns), None)
if img_col is None:
    raise ValueError(f"No image column found. Available columns: {df_raw.columns.tolist()}")

# 3) Pick angle column (common: 'steering_angle' or 'angle_norm')
possible_angle_cols = ["steering_angle", "angle", "angle_norm", "steer", "steering"]
angle_col = next((c for c in possible_angle_cols if c in df_raw.columns), None)
if angle_col is None:
    raise ValueError(f"No angle column found. Available columns: {df_raw.columns.tolist()}")

print(f"Using columns -> image: {img_col} | angle: {angle_col}")

# 4) Build clean dataframe
df = df_raw.copy()

# image filename only (fixes Windows full paths inside CSV)
df["image_path"] = df[img_col].astype(str).apply(os.path.basename)

# numeric angle
df["steering_angle"] = pd.to_numeric(df[angle_col], errors="coerce")

# drop bad rows
df = df.dropna(subset=["image_path", "steering_angle"]).reset_index(drop=True)

print("Clean rows:", len(df))
print(df.head(3)[["image_path", "steering_angle"]])


# (5)Verify Dataset files + preview sample rows

In [ ]:
import random

print("IMG_DIR exists:", IMG_DIR.exists())
if IMG_DIR.exists():
    print("IMG sample count:", len(list(IMG_DIR.glob('*.jpg'))))

print("CSV exists:", CSV_PATH.exists())
print("df rows:", len(df))

# If no rows yet, stop here nicely
if len(df) == 0:
    print("No data yet. Collect images + log.csv first, then rerun.")
else:
    sample = df.sample(min(5, len(df)), random_state=42)

    missing = 0
    for _, row in sample.iterrows():
        img_path = IMG_DIR / row["image_path"]   # filename inside IMG folder
        ok = img_path.exists()
        print(img_path.name, "->", ok, "| angle:", row["steering_angle"])
        if not ok:
            missing += 1

    print("Missing in sample:", missing)

# (6) Quick Visualisation

In [ ]:
import cv2
import matplotlib.pyplot as plt

def show_samples(n=6):
    if len(df) == 0:
        print("No data yet.")
        return

    samples = df.sample(min(n, len(df)), random_state=1)

    plt.figure(figsize=(12, 6))
    shown = 0

    for _, row in samples.iterrows():
        img_path = IMG_DIR / row["image_path"]
        angle = float(row["steering_angle"])

        img = cv2.imread(str(img_path))
        if img is None:
            continue

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        plt.subplot(2, 3, shown + 1)
        plt.imshow(img)
        plt.title(f"angle={angle:.3f}")
        plt.axis("off")

        shown += 1
        if shown >= 6:
            break

    plt.tight_layout()
    plt.show()

show_samples()

# (7) Angle Distribution

In [ ]:
import matplotlib.pyplot as plt

if len(df) == 0:
    print("No data yet.")
else:
    plt.figure(figsize=(8,4))
    plt.hist(df["steering_angle"], bins=30)
    plt.title("Steering Angle Distribution")
    plt.xlabel("Angle (norm: -1 to +1)")
    plt.ylabel("Count")
    plt.show()

    print(df["steering_angle"].describe())

# (8) Create PyTorch Dataset

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset

class SteeringDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, "image_path"]
        angle = float(self.df.loc[idx, "steering_angle"])

        img_path = self.img_dir / img_name
        img = cv2.imread(str(img_path))

        # If an image is missing/corrupt, return zeros (prevents crash)
        if img is None:
            x = torch.zeros((3, 66, 200), dtype=torch.float32)
            y = torch.tensor([angle], dtype=torch.float32)
            return x, y

        img = cv2.resize(img, (200, 66))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img = img.astype(np.float32) / 255.0
        img = (img - 0.5) / 0.5          # normalize to [-1, 1]
        img = np.transpose(img, (2, 0, 1)) # HWC -> CHW

        x = torch.from_numpy(img)
        y = torch.tensor([angle], dtype=torch.float32)
        return x, y

print("Dataset class ready ✅")

# (9) Train/Val Split + DataLoaders

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

if len(df) == 0:
    print("No data yet. Collect data first.")
else:
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)

    train_ds = SteeringDataset(train_df, IMG_DIR)
    val_ds   = SteeringDataset(val_df, IMG_DIR)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)

    xb, yb = next(iter(train_loader))
    print("x:", xb.shape, xb.dtype)
    print("y:", yb.shape, yb.dtype)
    print("y sample:", yb[:5].squeeze().tolist())

    print("Train samples:", len(train_ds))
    print("Val samples  :", len(val_ds))

# (10) Define CNN Model

In [ ]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 24, 5, stride=2), nn.ReLU(),
            nn.Conv2d(24, 36, 5, stride=2), nn.ReLU(),
            nn.Conv2d(36, 48, 5, stride=2), nn.ReLU(),
            nn.Conv2d(48, 64, 3), nn.ReLU(),
            nn.Conv2d(64, 64, 3), nn.ReLU(),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 1 * 18, 100), nn.ReLU(),
            nn.Linear(100, 50), nn.ReLU(),
            nn.Linear(50, 10), nn.ReLU(),
            nn.Linear(10, 1)
        )

    def forward(self, x):
        x = self.features(x)
        return self.regressor(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)

print("Device:", device)
print(model)

# (11) Loss + Optimizer

In [ ]:
# Fresh model each run
model = SimpleCNN().to(device)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_losses = []
val_losses = []

# (12) Training Loop

In [ ]:
import time
import torch

EPOCHS = 70

# If you haven't created loaders yet (because df is empty), keep them as None
train_loader = globals().get("train_loader", None)
val_loader   = globals().get("val_loader", None)

def run_epoch(loader, train=True):
    if loader is None:
        return None

    model.train() if train else model.eval()

    running = 0.0
    n = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        if train:
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
        else:
            with torch.no_grad():
                pred = model(x)
                loss = criterion(pred, y)

        running += loss.item() * x.size(0)
        n += x.size(0)

    return (running / n) if n > 0 else None


if train_loader is None or val_loader is None:
    print("No DataLoaders yet (no data). Training skipped for now.")
else:
    for epoch in range(EPOCHS):
        t0 = time.time()

        tr = run_epoch(train_loader, train=True)
        va = run_epoch(val_loader, train=False)

        train_losses.append(tr)
        val_losses.append(va)

        print(f"Epoch {epoch+1}/{EPOCHS} | train={tr:.4f} | val={va:.4f} | {time.time()-t0:.1f}s")

# (13) Plot Losses

In [ ]:
import matplotlib.pyplot as plt

if len(train_losses) == 0:
    print("No losses to plot yet (training not run).")
else:
    plt.figure(figsize=(7,4))
    plt.plot(train_losses, label="train")
    plt.plot(val_losses, label="val")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.show()

    print("Train losses:", train_losses)
    print("Val losses:", val_losses)

# (14) Save trained model (safe even if untrained)

In [ ]:
from pathlib import Path
import torch

# Save model weights (will still save even if you haven't trained yet)
MODEL_PATH = Path(DATA_DIR) / "steering_model.pth"
torch.save(model.state_dict(), MODEL_PATH)

print("Saved model to:", MODEL_PATH)

Optional but useful: Save the full checkpoint (model+optimizer+losses)

In [ ]:
CKPT_PATH = Path(DATA_DIR) / "steering_checkpoint.pth"
torch.save({
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "train_losses": train_losses,
    "val_losses": val_losses,
}, CKPT_PATH)

print("Saved checkpoint to:", CKPT_PATH)

# (15) Load model + inference

In [ ]:
import os
import cv2
import numpy as np
import torch
from pathlib import Path

# Change this if you saved somewhere else
MODEL_PATH = Path(DATA_DIR) / "steering_model.pth"   # same name as your save cell
TEST_IMAGE = None  # set later (we'll auto-pick one if IMG exists)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------- Load model weights ----------
if not MODEL_PATH.exists():
    print(f"❌ Model not found: {MODEL_PATH}")
    print("Run the training first (or at least run the save-model cell after defining the model).")
else:
    model = SimpleCNN().to(device)          # uses your class from earlier cell
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    print(f"✅ Loaded model: {MODEL_PATH}")

    # ---------- Pick an image to test ----------
    img_dir = Path(DATA_DIR) / "IMG"
    if TEST_IMAGE is None and img_dir.exists():
        # pick first jpg/png we can find
        candidates = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpeg"))
        if len(candidates) > 0:
            TEST_IMAGE = str(candidates[0])

    if TEST_IMAGE is None:
        print("⚠️ No test image found yet.")
        print(f"Put any image inside: {img_dir} then set TEST_IMAGE = r'full_path_to_image.jpg'")
    else:
        print("Testing on:", TEST_IMAGE)

        # ---------- Same preprocessing as dataset ----------
        img = cv2.imread(TEST_IMAGE)
        if img is None:
            print("❌ Could not read image. Check the file path.")
        else:
            img = cv2.resize(img, (200, 66))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            img = img.astype(np.float32) / 255.0
            img = (img - 0.5) / 0.5                 # normalize to [-1, 1]
            img = np.transpose(img, (2, 0, 1))      # CHW

            x = torch.from_numpy(img).unsqueeze(0).to(device)  # (1,3,66,200)

            with torch.no_grad():
                pred = model(x).item()

            print(f"✅ Predicted output: {pred:.4f}")

            # If your training label is angle_norm in [-1, 1], then pred is also in that space.
            # If you later want degrees, you'll need a mapping function based on your calibration.